### Feature Engineering

#### Importing Packages and Loading Data

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

In [2]:
project_root = Path.cwd().parent
processed_dir = project_root/"data"/"processed"

In [3]:
df = pd.read_parquet(processed_dir/"model_table.parquet")
df.head()

,ward_code,month,crime_type,crime_count,ward_name,borough
0,E05009288,2024-01,Anti-social behaviour,2,Aldersgate,City of London
1,E05009288,2024-02,Anti-social behaviour,2,Aldersgate,City of London
2,E05009288,2024-03,Anti-social behaviour,2,Aldersgate,City of London
3,E05009288,2024-04,Anti-social behaviour,0,Aldersgate,City of London
4,E05009288,2024-05,Anti-social behaviour,0,Aldersgate,City of London


In [4]:
df.shape

(168720, 6)

In [5]:
# Making another month date column
df["month_date"] = pd.to_datetime(df["month"], format="%Y-%m")

In [6]:
# Sorting so each ward and crime type runs in true time order
df = df.sort_values(["ward_code", "crime_type", "month_date"]).reset_index(drop=True)

#### Lag Features

In [8]:
group = df.groupby(["ward_code", "crime_type"])

In [10]:
# Last month, 2 months ago, 3 months ago and same month previous year
df["lag_1"] = group["crime_count"].shift(1)
df["lag_2"] = group["crime_count"].shift(2)
df["lag_3"] = group["crime_count"].shift(3)
df["lag_12"] = group["crime_count"].shift(12)

df.head()

,ward_code,month,crime_type,crime_count,ward_name,borough,month_date,lag_1,lag_2,lag_3,lag_12
0,E05009288,2024-01,Anti-social behaviour,2,Aldersgate,City of London,2024-01-01,NaN,NaN,NaN,NaN
1,E05009288,2024-02,Anti-social behaviour,2,Aldersgate,City of London,2024-02-01,2.0,NaN,NaN,NaN
2,E05009288,2024-03,Anti-social behaviour,2,Aldersgate,City of London,2024-03-01,2.0,2.0,NaN,NaN
3,E05009288,2024-04,Anti-social behaviour,0,Aldersgate,City of London,2024-04-01,2.0,2.0,2.0,NaN
4,E05009288,2024-05,Anti-social behaviour,0,Aldersgate,City of London,2024-05-01,0.0,2.0,2.0,NaN


#### Rolling Mean

In [14]:
# Average of previous 3 months
df["roll_mean_3"] = (
    group["crime_count"]
    .shift(1)
    .rolling(window=3, min_periods=1)
    .mean()
    .reset_index(drop=True)
)
df.head()

,ward_code,month,crime_type,crime_count,ward_name,borough,month_date,lag_1,lag_2,lag_3,lag_12,month_num,is_summer,roll_mean_3
0,E05009288,2024-01,Anti-social behaviour,2,Aldersgate,City of London,2024-01-01,NaN,NaN,NaN,NaN,1,0,NaN
1,E05009288,2024-02,Anti-social behaviour,2,Aldersgate,City of London,2024-02-01,2.0,NaN,NaN,NaN,2,0,2.000000
2,E05009288,2024-03,Anti-social behaviour,2,Aldersgate,City of London,2024-03-01,2.0,2.0,NaN,NaN,3,0,2.000000
3,E05009288,2024-04,Anti-social behaviour,0,Aldersgate,City of London,2024-04-01,2.0,2.0,2.0,NaN,4,0,2.000000
4,E05009288,2024-05,Anti-social behaviour,0,Aldersgate,City of London,2024-05-01,0.0,2.0,2.0,NaN,5,0,1.333333


#### Calender Features and Seasonality

In [15]:
df["month_num"] = df["month_date"].dt.month
df["is_summer"] = df["month_num"].isin([6,7,8]).astype(int)

In [16]:
df

,ward_code,month,crime_type,crime_count,ward_name,borough,month_date,lag_1,lag_2,lag_3,lag_12,month_num,is_summer,roll_mean_3
0,E05009288,2024-01,Anti-social behaviour,2,Aldersgate,City of London,2024-01-01,NaN,NaN,NaN,NaN,1,0,NaN
1,E05009288,2024-02,Anti-social behaviour,2,Aldersgate,City of London,2024-02-01,2.0,NaN,NaN,NaN,2,0,2.000000
2,E05009288,2024-03,Anti-social behaviour,2,Aldersgate,City of London,2024-03-01,2.0,2.0,NaN,NaN,3,0,2.000000
3,E05009288,2024-04,Anti-social behaviour,0,Aldersgate,City of London,2024-04-01,2.0,2.0,2.0,NaN,4,0,2.000000
4,E05009288,2024-05,Anti-social behaviour,0,Aldersgate,City of London,2024-05-01,0.0,2.0,2.0,NaN,5,0,1.333333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
168715,E05014119,2025-08,Violence and sexual offences,14,West Dulwich,Lambeth,2025-08-01,18.0,19.0,16.0,20.0,8,1,17.666667
168716,E05014119,2025-09,Violence and sexual offences,17,West Dulwich,Lambeth,2025-09-01,14.0,18.0,19.0,16.0,9,0,17.000000
168717,E05014119,2025-10,Violence and sexual offences,15,West Dulwich,Lambeth,2025-10-01,17.0,14.0,18.0,15.0,10,0,16.333333
168718,E05014119,2025-11,Violence and sexual offences,24,West Dulwich,Lambeth,2025-11-01,15.0,17.0,14.0,18.0,11,0,15.333333


In [17]:
# A forecast needs at least 1 month of history and require lag_1
df = df.dropna(subset=["lag_1"]).reset_index(drop=True)
df

,ward_code,month,crime_type,crime_count,ward_name,borough,month_date,lag_1,lag_2,lag_3,lag_12,month_num,is_summer,roll_mean_3
0,E05009288,2024-02,Anti-social behaviour,2,Aldersgate,City of London,2024-02-01,2.0,NaN,NaN,NaN,2,0,2.000000
1,E05009288,2024-03,Anti-social behaviour,2,Aldersgate,City of London,2024-03-01,2.0,2.0,NaN,NaN,3,0,2.000000
2,E05009288,2024-04,Anti-social behaviour,0,Aldersgate,City of London,2024-04-01,2.0,2.0,2.0,NaN,4,0,2.000000
3,E05009288,2024-05,Anti-social behaviour,0,Aldersgate,City of London,2024-05-01,0.0,2.0,2.0,NaN,5,0,1.333333
4,E05009288,2024-06,Anti-social behaviour,0,Aldersgate,City of London,2024-06-01,0.0,0.0,2.0,NaN,6,1,0.666667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
161685,E05014119,2025-08,Violence and sexual offences,14,West Dulwich,Lambeth,2025-08-01,18.0,19.0,16.0,20.0,8,1,17.666667
161686,E05014119,2025-09,Violence and sexual offences,17,West Dulwich,Lambeth,2025-09-01,14.0,18.0,19.0,16.0,9,0,17.000000
161687,E05014119,2025-10,Violence and sexual offences,15,West Dulwich,Lambeth,2025-10-01,17.0,14.0,18.0,15.0,10,0,16.333333
161688,E05014119,2025-11,Violence and sexual offences,24,West Dulwich,Lambeth,2025-11-01,15.0,17.0,14.0,18.0,11,0,15.333333


In [20]:
# Filling remaning lag NaNs with 0
lag_cols = ["lag_1", "lag_2", "lag_3", "lag_12", "roll_mean_3"]
df[lag_cols] = df[lag_cols].fillna(0)
df.head()

,ward_code,month,crime_type,crime_count,ward_name,borough,month_date,lag_1,lag_2,lag_3,lag_12,month_num,is_summer,roll_mean_3
0,E05009288,2024-02,Anti-social behaviour,2,Aldersgate,City of London,2024-02-01,2.0,0.0,0.0,0.0,2,0,2.000000
1,E05009288,2024-03,Anti-social behaviour,2,Aldersgate,City of London,2024-03-01,2.0,2.0,0.0,0.0,3,0,2.000000
2,E05009288,2024-04,Anti-social behaviour,0,Aldersgate,City of London,2024-04-01,2.0,2.0,2.0,0.0,4,0,2.000000
3,E05009288,2024-05,Anti-social behaviour,0,Aldersgate,City of London,2024-05-01,0.0,2.0,2.0,0.0,5,0,1.333333
4,E05009288,2024-06,Anti-social behaviour,0,Aldersgate,City of London,2024-06-01,0.0,0.0,2.0,0.0,6,1,0.666667


In [21]:
# Saving
out_path = processed_dir / "model_features.parquet"
df.to_parquet(out_path, index=False)

### Summary

Lag features: For every (ward, crime type) series, Counts from 1, 2, 3, and 12 months ago were built. The 1–3 month lags capture short-term momentum and the 12-month lag captures the seasonal pattern.

Rolling mean: A 3-month rolling average smooths out the noise of any
single month.

Calendar features:  Month number (1–12) was added and a binary flag for the summer months (June–August), the period EDA showed has the strongest crime surge.

The very first month of each series has no prior data to lag, so those rows were dropped. The remaining lag NaNs were filled with 0.